In [2]:
# Re-import necessary packages after kernel reset
import cv2
import os
import numpy as np
import pandas as pd
from PIL import Image

input_folder = "C:\\Users\\ysang\\Downloads\\DATA-20250603T024153Z-1-001\DATA"
output_csv = "C:\\Users\\ysang\\Downloads\\facial_region_features.csv"

# Load Haar cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Prepare output storage
results = []

# Scan all image files in the folder
valid_extensions = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_extensions)]

# Process each image
for filename in image_files:
    image_path = os.path.join(input_folder, filename)
    img = cv2.imread(image_path)

    if img is None:
        continue  # Skip if image failed to load

    # Resize small images for better face detection (optional for speed)
    if img.shape[1] < 600:
        img = cv2.resize(img, (600, int(img.shape[0] * (600 / img.shape[1]))))

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    if len(faces) == 0:
        continue  # Skip images with no detected face

    for (x, y, w, h) in faces:
        # Define cheek regions
        left_cheek = gray[y + h//2 : y + int(0.85*h), x + int(0.1*w) : x + int(0.4*w)]
        right_cheek = gray[y + h//2 : y + int(0.85*h), x + int(0.6*w) : x + int(0.9*w)]

        for region, label in zip([left_cheek, right_cheek], ['Left Cheek', 'Right Cheek']):
            if region.size == 0:
                continue
            mean_intensity = np.mean(region)
            contrast = np.std(region)
            laplacian_var = cv2.Laplacian(region, cv2.CV_64F).var()

            results.append({
                "Image": filename,
                "Region": label,
                "Mean Intensity": mean_intensity,
                "Contrast": contrast,
                "Texture (Laplacian Var)": laplacian_var
            })

# Save to CSV
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)

df.head()


,Image,Region,Mean Intensity,Contrast,Texture (Laplacian Var)
0,1.JPG,Left Cheek,106.112457,25.945389,148.661864
1,1.JPG,Right Cheek,105.720970,29.233683,124.940042
2,1.JPG,Left Cheek,152.706349,16.442975,990.664903
3,1.JPG,Right Cheek,145.312169,15.972460,1070.280542
4,1.JPG,Left Cheek,131.065476,13.533640,1003.495675


In [4]:
df.groupby(['Image', 'Region']).mean().reset_index()


,Image,Region,Mean Intensity,Contrast,Texture (Laplacian Var)
0,1.JPG,Left Cheek,129.961428,18.640668,714.274147
1,1.JPG,Right Cheek,124.767991,18.980707,498.887773
2,10.JPG,Left Cheek,145.422373,26.816390,140.307711
3,10.JPG,Right Cheek,157.101949,16.390687,71.707069
4,100.JPG,Left Cheek,187.757843,12.116292,104.687805
...,...,...,...,...,...
593,97.JPG,Right Cheek,154.840106,14.527741,45.069269
594,98.JPG,Left Cheek,111.396360,35.406295,136.288416
595,98.JPG,Right Cheek,104.696967,32.163373,104.323901
596,99.JPG,Left Cheek,150.628069,24.869046,167.141990


,Region,Mean Intensity,Contrast
0,Left Cheek,156.630119,26.097104
1,Right Cheek,155.745160,25.159728
